In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from scraper import fetch_website_contents

In [3]:
class CompetitorAnalysis(BaseModel):
    company_name: str = Field(description="Name of the company or product")
    primary_offering: str = Field(description="Core service or value proposition")
    strengths: List[str] = Field(description="3 to 5 notable strengths, advantages, or standout features")
    weaknesses: List[str] = Field(description="3 to 5 potential weaknesses, limitations, missing features, or pricing/transparency gaps")

llm = ChatOllama(model="deepseek-r1:1.5b", temperature=0).with_structured_output(CompetitorAnalysis)

In [4]:
system_message = """ You are an expert market intelligence and product analyst. \
Analyze the provided website content and extract the company's core offering, strengths, and weaknesses based strictly on the text provided."""

In [5]:
template = ChatPromptTemplate.from_messages([
    ('system', '{system_message}'),
    ('human', 'Website URL: {url}\n\nWebsite Content:\n{content}')
])

In [6]:
chain = template.partial(system_message=system_message) | llm

In [7]:
urls = [
    "https://realpython.com",
    "https://www.boot.dev",
    "https://exercism.org/tracks/python",
    "https://www.learnpython.org",
]

In [8]:
reports: CompetitorAnalysis = []
for url in urls:
    content = fetch_website_contents(url)

    response = chain.invoke({"url": url, "content": content})
    reports.append(response)

In [9]:
def format(report: CompetitorAnalysis):
    print("Company name: " + report.company_name + '\n')
    print("Primary_offering: " + report.primary_offering)

    print("\nStrengths:")
    for i, strenght in enumerate(report.strengths, start=1):
        print(f"{i}. {strenght}")

    print("\nWeaknesses:")
    for i, weakness in enumerate(report.weaknesses, start=1):
            print(f"{i}. {weakness}")

In [10]:
for i, report in enumerate(reports, start=1):
    print(f"Website {i}:-\n")
    format(report)
    print("\n\n")

Website 1:-

Company name: Real Python

Primary_offering: Comprehensive Python tutorials and resources

Strengths:
1. Extensive tutorials that make learning Python enjoyable and effective
2. Live Python courses that provide hands-on experience
3. Value newsletter that keeps readers informed about new features and updates

Weaknesses:
1. Improvements needed in mobile experience and language-specific resources



Website 2:-

Company name: Boot.dev

Primary_offering: Learn to Code | Boot.dev

Strengths:
1. Clear and structured content focusing on coding education.
2. Practical advice on coding, avoiding tutorial hell, and using a game-like curriculum.
3. Attractive pricing model offering 1% the price of college for discounted access.
4. Targeted for businesses and students, providing relevant content.

Weaknesses:
1. Potential focus on coding education only, which may limit broader technical areas.
2. Technical language and dense content that may appeal more to developers.
3. Lack of eng